# Módulo 6.2 — Solución: Limpieza Completa

Resolución paso a paso del **Caso Práctico** de la clase 6.2 (`Carga de Archivos y Limpieza de Datos`).

Partimos de `estudiantes_crudo.csv` (sin limpiar) y lo dejamos listo para análisis, guardándolo como `estudiantes_limpio.csv`.

## Paso 1: Cargar el archivo y revisar `.info()`

El archivo usa `;` como separador (típico de un CSV exportado desde Excel en español). Con `.info()` vemos de una vez los tipos de dato y cuántos valores no nulos tiene cada columna.

In [1]:
import pandas as pd

df = pd.read_csv('estudiantes_crudo.csv', sep=';')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   nombre      11 non-null     object 
 1   carrera     11 non-null     object 
 2   semestre    11 non-null     int64  
 3   nota        10 non-null     float64
 4   asistencia  11 non-null     int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 568.0+ bytes


In [2]:
df

,nombre,carrera,semestre,nota,asistencia
0,Ana,Ingeniería,3,8.5,95
1,Luis,administración,5,7.2,80
2,Marcela,Medicina,2,9.1,98
3,Pedro,INGENIERÍA,3,NaN,70
4,Sofía,Derecho,6,8.0,88
5,Diego,Administración,4,7.5,92
6,Valentina,medicina,2,9.4,99
7,Carlos,Arquitectura,5,6.5,65
8,Isabella,Ingeniería,1,8.9,90
9,Andrés,Derecho,6,7.8,85


`nota` tiene 10 valores no nulos de 11 filas — ya sabemos que ahí hay un nulo. También se nota, a simple vista, que `carrera` tiene valores inconsistentes (`'Ingeniería'`, `'administración '`, `'INGENIERÍA'`, `' medicina'`) y que la última fila repite a Ana.

## Paso 2: Contar los nulos por columna y ver las filas afectadas

Primero contamos cuántos nulos hay por columna con `isna().sum()`, y luego filtramos para ver exactamente **cuáles filas** tienen el problema antes de decidir qué hacer.

In [3]:
df.isna().sum()

nombre        0
carrera       0
semestre      0
nota          1
asistencia    0
dtype: int64

In [4]:
df[df['nota'].isna()]

,nombre,carrera,semestre,nota,asistencia
3,Pedro,INGENIERÍA,3,NaN,70


Solo `nota` tiene un valor faltante, en la fila de Pedro (índice 3).

## Paso 3: Rellenar la nota faltante con el promedio

Usamos `.fillna()` con el promedio de la columna — una estrategia razonable cuando solo falta un valor aislado y no queremos perder toda la fila.

In [5]:
df['nota'] = df['nota'].fillna(df['nota'].mean())
df.loc[[3]]

,nombre,carrera,semestre,nota,asistencia
3,Pedro,INGENIERÍA,3,8.14,70


La nota de Pedro ya no es `NaN`: quedó en el promedio del grupo (8.14).

## Paso 4: Mostrar las filas duplicadas y luego eliminarlas

Antes de borrar nada, confirmamos **cuáles** filas son duplicadas con `df[df.duplicated()]`. Recién después las eliminamos con `.drop_duplicates()`.

In [6]:
df[df.duplicated()]

,nombre,carrera,semestre,nota,asistencia
10,Ana,Ingeniería,3,8.5,95


In [7]:
filas_antes = len(df)
df = df.drop_duplicates().copy()
filas_despues = len(df)

print(f'Filas antes: {filas_antes}  →  Filas después: {filas_despues}')

Filas antes: 11  →  Filas después: 10


La copia repetida de Ana (fila 10) desapareció: pasamos de 11 a 10 filas.

## Paso 5: Limpiar la columna `carrera`

Quitamos espacios en blanco con `.str.strip()` y normalizamos mayúsculas/minúsculas con `.str.lower()`, para que `'Ingeniería'`, `'INGENIERÍA'` y `' ingeniería '` queden como una sola categoría.

In [8]:
df['carrera'].unique()

array(['Ingeniería', 'administración ', 'Medicina', 'INGENIERÍA',
       'Derecho', 'Administración', ' medicina', 'Arquitectura'],
      dtype=object)

In [9]:
df['carrera'] = df['carrera'].str.strip().str.lower()
df['carrera'].unique()

array(['ingeniería', 'administración', 'medicina', 'derecho',
       'arquitectura'], dtype=object)

Antes de limpiar, `'ingeniería'` y `'administración'` aparecían con varias formas distintas. Después de `.str.strip().str.lower()`, cada carrera queda en una sola categoría.

## Paso 6: Renombrar `nota` a `calificacion`

In [10]:
df = df.rename(columns={'nota': 'calificacion'})
df.columns.tolist()

['nombre', 'carrera', 'semestre', 'calificacion', 'asistencia']

## Paso 7: Eliminar la columna `semestre`

Asumimos que no la necesitamos para este análisis.

In [11]:
df = df.drop(columns=['semestre'])
df.columns.tolist()

['nombre', 'carrera', 'calificacion', 'asistencia']

## Paso 8: Guardar el resultado en `estudiantes_limpio.csv`

Usamos `index=False` para no guardar la columna de índice numérico como si fuera una columna más de datos.

In [12]:
df['calificacion'] = df['calificacion'].round(2)
df.to_csv('estudiantes_limpio.csv', index=False)
df

,nombre,carrera,calificacion,asistencia
0,Ana,ingeniería,8.50,95
1,Luis,administración,7.20,80
2,Marcela,medicina,9.10,98
3,Pedro,ingeniería,8.14,70
4,Sofía,derecho,8.00,88
5,Diego,administración,7.50,92
6,Valentina,medicina,9.40,99
7,Carlos,arquitectura,6.50,65
8,Isabella,ingeniería,8.90,90
9,Andrés,derecho,7.80,85


## Verificación final

Volvemos a leer el archivo guardado para confirmar que quedó tal como esperábamos: sin nulos, sin duplicados, `carrera` normalizada, y con las columnas correctas.

In [13]:
verificacion = pd.read_csv('estudiantes_limpio.csv')

print('Columnas:', verificacion.columns.tolist())
print('Nulos por columna:')
print(verificacion.isna().sum())
print('Duplicados:', verificacion.duplicated().sum())
print('Carreras únicas:', verificacion['carrera'].unique())
verificacion

Columnas: ['nombre', 'carrera', 'calificacion', 'asistencia']
Nulos por columna:
nombre          0
carrera         0
calificacion    0
asistencia      0
dtype: int64
Duplicados: 0
Carreras únicas: ['ingeniería' 'administración' 'medicina' 'derecho' 'arquitectura']


,nombre,carrera,calificacion,asistencia
0,Ana,ingeniería,8.50,95
1,Luis,administración,7.20,80
2,Marcela,medicina,9.10,98
3,Pedro,ingeniería,8.14,70
4,Sofía,derecho,8.00,88
5,Diego,administración,7.50,92
6,Valentina,medicina,9.40,99
7,Carlos,arquitectura,6.50,65
8,Isabella,ingeniería,8.90,90
9,Andrés,derecho,7.80,85
